# Drag prediction model

This notebook demonstrates a machine learning model geared towards predicting atmospheric drag given solar weather parameters. One dataset is taken from ESA's [Swarm project](https://swarm-diss.eo.esa.int/#swarm/Multimission), and the other is GFZ's [space weather](https://www-app3.gfz-potsdam.de/kp_index/Kp_ap_Ap_SN_F107_since_1932.txt) data, dating back to 1934. Together, the data provide a 10-second resolution dataset comprising spatiotemporal electron density, solar flux, sunspot number, and geomagnetic disturbances.

### Identify all CDF data files

In [1]:
data_dir = './data/'
cdf_paths = data_dir + '**/**/**/**/*.cdf'
k_a_f107_path = data_dir + 'Kp_ap_Ap_SN_F107_since_1932.txt'

### Read / amalgamate all CDF files

In [2]:
import sys
import glob
import numpy as np
import pandas as pd

from src.utilities.file_helper import read_cdf

data_files = glob.glob(cdf_paths)

ex_df = read_cdf(data_files[0])

### Read solar weather data

In [3]:
from src.utilities.file_helper import read_f107_file

# Read atmospheric data file (daily since 1934)
df = read_f107_file(k_a_f107_path)

# Create the date column
df['date'] = pd.to_datetime(dict(year=df.Year, month=df.Month, day=df.Day))

### Merge solar weather with atmospheric drag data

In [4]:
# Extract date from GRACE timestamps for merging
ex_df['date'] = ex_df['time'].dt.normalize()

# Merge on date
merged_df = pd.merge(ex_df, df, on='date')

# Then pick the right kp/ap column based on hour
ex_df['hour'] = ex_df['time'].dt.hour
interval = (ex_df['hour'] // 3) + 1  # 1-8 for the 8 three-hour intervals

# Map to correct kp/ap column
merged_df['kp'] = merged_df.apply(lambda r: r[f'kp{int((r["time"].hour // 3) + 1)}'], axis=1)
merged_df['ap'] = merged_df.apply(lambda r: r[f'ap{int((r["time"].hour // 3) + 1)}'], axis=1)

## Neural network

### Data cleaning

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Drop anomalous data
merged_df = merged_df[merged_df['validity_flag']==1]

# Handle missing values (example: drop rows with missing values)
merged_df.dropna(inplace=True)
merged_df.drop(columns=['time','local_solar_time','density_orbitmean','validity_flag','validity_flag_orbitmean'])

# Separate features (X) and target variable (y)
target_variable = 'density_kg_m3'
X = merged_df.drop(target_variable, axis=1)
y = merged_df[target_variable]

# Convert categorical columns to numerical using one-hot encoding (if applicable)
X = pd.get_dummies(X, drop_first=True)

# Convert pandas DataFrames to NumPy arrays, which is the required input format for neural networks
X_np = X.values
y_np = y.values

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

# Scale features (important for most neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Define / train network

In [ ]:
# Initialize a sequential model
model = Sequential()

# Add layers
# Input layer (implicitly defined by the input_dim of the first dense layer)
model.add(Dense(units=32, activation='relu', input_dim=X_train_scaled.shape[1])) # Use the number of features as input dimension
# Hidden layer
model.add(Dense(units=16, activation='relu'))
# Output layer (1 neuron for binary classification, use 'sigmoid' activation)
model.add(Dense(units=1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', # Use binary_crossentropy for binary classification
              optimizer='adam',
              metrics=['accuracy'])

# Train the model
model.fit(X_train_scaled, y_train, epochs=50, batch_size=10, verbose=1, validation_split=0.2)

# Evaluate the model
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f'Test Accuracy: {accuracy*100:.2f}%')

# Make predictions
predictions = model.predict(X_test_scaled)
# Convert predictions to binary classes (0 or 1)
predictions = (predictions > 0.5).astype(int)